# Lecture 5: Linear Models

Welcome to your first class on machine learning algorithms! Today, we will look at the easiest family of algorithms to understand: linear models. We will also learn the difference between different machine learning approaches: supervised and unsupervised algorithms. Finally, we will look at regression versus classification in supervised machine learning. 

You will find that the term 'linear model' encompasses many more concepts than you may have expected. Of course, the ordinary least squares algorithm (which you learned about in school) is a linear model. But so are gradient descent algorithms and the logistic regression classifiers.

Because they are fairly simple, linear models are often a good place to start when you explore a new dataset and want to predict the value of a numeric property or a category in future unknown data.

## Learning Objectives
1. Articulate the different approaches in machine learning
2. Understand how linear regression is a type of machine learning algorithm
3. Perform multi-linear regression tasks with ordinary least square and SGDRegressors
4. Perform classification tasks with logistic regression

## 5.0 Introduction: Different Types of Machine Learning Applications
<a id="intro"></a>

In [ ]:
from IPython.display import IFrame
display(IFrame('https://youtube.com/embed/wZc0N3TOX_Y', width = 640, height = 360))

### 5.0.1 Setting up the notebook
<a id="setup"></a>

In this section, we will largely reuse the data from the previous lectures. So, our first step will be to load our dataset. Of course, the *very* first thing we will want to do is import `numpy`, `pandas` and `matplotlib`. If you want, you can also do some pre-formatting, which will affect the plots you generate throughout the rest of the notebook.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# To plot pretty figures <- this merely sets some parameters that will be applied to all plots in this notebook.
# You can adjust this to whatever you want - you can even look through the documentation for more customizable options
mpl.rc('axes', labelsize=14) # Change the label text size on the axis name
mpl.rc('xtick', labelsize=12) # Change the tick text size on the x-axis
mpl.rc('ytick', labelsize=12) # Change the tick text size on the y-axis

# Exercise 1

Here we will use simply the dataset of porosity vs NGR and %Carbonate that we used before. This is a simplified version of the dataset, and it contains no labels (only numerical values are included). The data file is saved in 'Datasets/core_data.csv'.

You are now an expert at data preparation, so do the following for the above dataset:

* Remove duplicates
* Separate the target (`Porosity (vol%)` - call it `y`) from the features (call them `X`)
* Split the dataset into 70% training, 30% testing. To ensure we get the same results use a `random_state` value of 42.
* Using a pipeline, impute missing values using a `SimpleImputer` class with a `mean` strategy and a `MinMaxScaler`

You should know how to do all of the above except for splitting the target and the features. You can use the brief code block below for inspiration:

```python
# Create features (drop the target columns)
X = data.drop(columns = 'Porosity (vol%)')

# Create target
y = data['Porosity (vol%)'] 

# Split the data using train/test split. Note that when you provide two dataframes (X and y), it will split them up 
# in the order shown
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.7, random_state = 42)
```

After applying your data preparation pipeline, you should have the prepared predictors (`X_train_prep` and `X_test_prep`) and the unchanged targets (`y_train` and `y_test`). The prepared predictors have missing values filled and features scaled. Keep the original `X_train` and `X_test` unchanged for the later exercises.

**Tip 1:** If you get stuck, you can refer to last week's exercise (this is essentially the same workflow). That said, I do recommend that you try and go as far as you can on your own: it is important to practice to be able to do it without the need to refer to previous code...

**Tip 2:** The `sklearn` `Pipeline` class will return a Numpy Array, not a Pandas Dataframe. For many applications, this is ok as you no longer need to explore your dataset once it has been prepared; you simply need to send it to another `sklearn` object, which accept numpy arrays without complaint. But in our case, we will want to convenience of calling specific columns later. So make sure that both `X_train` and `X_test` are Dataframes (you can refer to the documentation to see how to turn Numpy arrays directly into DataFrames). 

In [ ]:
# YOUR CODE HERE


## 5.1 Linear regression 
<a id="linreg"></a>

In [ ]:
from IPython.display import IFrame
display(IFrame('https://youtube.com/embed/plyX3n7a0lM', width = 640, height = 360))

### 5.1.1 Linear Regression using "Normal" Equations

The concept of linear regression shouldn't be new to you - some of you may even have been exposed to the concept of linear regression since secondary school. At the very least, you should know the formula for a linear equation, which is a closed solution to the regression problem. But what you may not have realized is that linear regression is the simplest type of machine learning algorithm!

If you think about it, regression has all the hallmarks of a machine learning algorithm:

1. It generates a solution based on the data you give to the model, not on any physics-based model
2. The guiding principle of the linear regression algorithm is to reduce the distance between each datapoint and our linear regression model - in machine learning-speak, this is known as a **cost function**. Many other machine learning algorithms use a cost function to help find the best fit to the model.

In the traditional, simple linear models that you are familiar with, where there are generally not too many datapoints, there exists an equation with a closed solution (i.e., an equation that can be determined directly). You are probably familiar with the two-dimensional univariate (i.e. one variable) version of this equation:

\begin{align}
    y &= b + aX \quad &\text{(Eq. 1)}
\end{align}

However, we can generalise Equation 1 by rewriting it in the following form:

\begin{align}
h(\theta) &=\theta_0X_0 + \theta_1X_1 \quad &\text{(Eq. 2)}
\end{align}

You can see that Equation 1 is simply a specifc case of Equation 2 where $\theta_0=1$ and $X_0=b$. What is interesting is that we can generalise this further by adding dimensions to obtain a multivariate regression equation:

\begin{align}
h(\theta) =\theta_0X_0 + \theta_1X_1 + ... + \theta_nX_n \quad &\text{(Eq. 3)}
\end{align}

This is quite confusing, and you might be wondering about how it applies to machine learning. Think of it this way: the normal linear equation that you're familiar with finds the best-fit relationship between two variables. The *multivariate* regression equation does the same thing but for as many variables as you need.

This is particularly relevant since all real-world datasets are multidimensional! Our own sedimentary core dataset contains 5 feature vectors (Depth, NGR, a*, b*, and L*), so it is of dimension 5. The good news is that we can use all of the feature vectors and solve for them using the linear regression equations presented above. You can do this manually using matrix operations, but neither you nor I have the time for that, and it is far beyond the scope of this introduction. Instead, we can exploit the Scikit-learn ecosystem to apply a linear regression algorithm to our dataset by fitting a `LinearRegression()` predictor. 

### Using Linear Regression in Python

Before we do that though, let's review how we use a `Pipeline` in `sklearn`:

* First, we need to `fit` the pipeline using ```pipeline.fit(X_train) ``` . When we `fit` a pipeline it effectively learns the statistical parameters of the model based on the `X_train` dataset. So far, these have been easy descriptive statistical terms, such as the means of each column.
* Then, we can transform any given dataset by calling the `transform` method of our previously fitted pipeline.

This logic of `fit` first and then `transform` also applies to machine learning algorithms in `sklearn`. When you `fit` an ordinary least square model (`LinearRegression()`) the object is learning the best parameters of the `X_train` dataset. In the case of a `LinearRegression()` model, these parameters represent the slope and intercept of the equation.

`LinearRegression()` is an example of a **supervised** algorithm. There are two things that must be passed to the `.fit` method of supervised algorithms:  the features (`X_train`) and the labels or solutions for each of those records (`y_train`).

`sklearn` machine learning models introduce a new method that tranformers (such as `MinMaxScaler`) don't have: the `predict` method. Once fitted, the `.predict` method allows the model to make predictions about new data (in our case, the `X_test`). The predicted results can then be compared to `y_test` to get an idea of how well our model would perform on unknown data.

Alright, let's train our first machine learning model on the dataset you prepared above:

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()

lin_reg.fit(X_train_prep, y_train)

We can now inspect the trained model object and query the different parameters that have been fitted based on the dataset: the intercept and coefficient:

In [ ]:
lin_reg.intercept_, lin_reg.coef_

As you can see, there is one intercept, and 5 coefficients: one coefficient per feature we used to predict porosity
 - Depth
 - NGR
  - L*
  - a*
  - b*.

Now that we have our model, how can we use it to predict on our X_test values?

In [ ]:
y_pred = lin_reg.predict(X_test_prep)
y_pred

Congratulations! You have now successfully predicted the porosity of 30% of your original dataset. Obviously, not the end of the exercise - after all, we have no idea if our predicted porosities are any good! We'll be exploring this in Exercise 2.

# Exercise 2: Incorporating ML models in pipelines

By now you should be somewhat familiar with pipelines since you've been using them to prepare your data since the last lecture. But the real power of pipelines is that they allow you to pair data preparation and model training into a single pipeline. All you have to do is simply add a predictor (i.e., a model) at your last step in our pipeline. Then, to assess our model, all we need to do is:

* `fit` the pipeline using the original training predictors and their target values: `ols_pipe.fit(X_train, y_train)`.
* Predict porosity from the original test predictors: `y_pred = ols_pipe.predict(X_test)`.

Note that you no longer need to save the intermediate transformed `X_train` and `X_test` dataframe: the `transform` method is called under the hood by the pipeline for you.

In the following cell, redo Exercise 1 but this time incorporate a `LinearRegression()` model into your pipeline (at the end), `fit` it, and create a `y_pred` prediction from your test set. Call your pipeline `ols_pipe` (for 'ordinary least square'): we will reuse it in the rest of this notebook.

**Tip:** You will need to resplit the data as the current `X_train` and `X_test` are already scaled, but you can copy most of your code from Exercise 1 so this will be quick.

In [ ]:
# Your code here


If you did this correctly, the `y_pred` you got from your pipeline should be identical to the `y_pred` you generated from the previous workflow. By using a pipeline, you've successfully simplified the process and used less code!

But this still doesn't answer the question of how well our model is performing. The beauty of using a test set is that we know the correct answer: we have an actual measure of porosity in these sediments. What we need now is a metric to evaluate our model.

## 5.1.2 Regression Metrics 

**Beware**: Math ahead!

There are several metrics that can be used to evaluate a regression model, and each have own their benefits and limitations. First, let's review the different metrics.

Below, $y_i$ represents the actual label (value) of sample $i$ (in our case above, `y_test`), and $\hat y_i$ the value of $i$ predicted by our model (above, `y_pred`). 

We are trying to reduce the errors made by our model; in most cases, the lower the metric, the better. But for the coefficient of determination, it is the reverse: the higher $R^2$ is, the better.

### Mean Squared Error (MSE)
$$\text{MSE} = {\frac{1}{n} \sum_{i = 1}^{n} (y_i - \hat y_i)^2}$$

### Root Mean squared error (RMSE)
* The units of the error are in the same units as the target variable
$$\text{RMSE} = \sqrt{ \frac{1}{n} \sum_{i = 1}^{n} (y_i - \hat y_i)^2 }$$


### Mean absolute error
* Less sensitive to outliers (errors from outliers are squared in RMSE, increasing error)
* The units of the error are in the same units as the target variable
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y_i}|$$


### Max Error
$$ ME = \max_{i=1}^{n} |y_i - \hat{y_i}|$$


### Coefficient of determination ($R^2$)
$$ R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y_i})^2}{\sum_{i=1}^{n}(y_i - \overline{y})^2}$$

So which metric should you choose to assess your regression model? 

There is no one size fits all when you evaluate models. A good place to start though is here:
* 👉 Use MSE when you need to penalize large errors and when the sign of the error is not important (i.e., when a positive error and a negative error are the same). 
* 👉 Use RMSE when you want to penalize large errors while ensuring that the errors are in the same unit as the target variable.
* 👉 Use MAE when all errors, large or small, have equal importance.
* 👉 Use Max Error when you want to limit the magnitude of the errors.
* 👉 Use $R^2$ when you want a general/comparable performance metric and where units are not required.


🚨 Note that there is no standardised value for Mean Square Error, Root Mean Square Error, and Mean Absolute Error. Scores will be relative to the magnitude of the units within the dataset. So a direct comparison of RMSE between models that are trained to predict different propertes is meaningless. Another way to think about it is this: half a million year is not a bad error on a U/Pb radiometric age, but if you are half a million grams off when measuring the concentration of salt on your lab bench, something is very wrong!

### 💻 Metrics in Sklearn

You can use metrics in `sklearn` very easily by importing them from the `metrics` module. All you need is a `y_pred` and a `y_test`.

Here is an example using the `mean_squared_error` and the `y_pred` you produced in Exercise 2:

In [ ]:
from sklearn.metrics import mean_squared_error
import math # to nicely round my outputs

mse = mean_squared_error(y_test, y_pred)

print('MSE =', round(mse, 2))

# Exercise 3

Following the example above, now calculate the MSE, RMSE, MAE, Max Error and R^2 for your model, and print the results neatly.

**Tip:** You will need to check the `sklearn` documentation to be able to find the names of the different metrics. 

In [ ]:
# Your code

### 5.1.3 Linear regression using Gradient Descent
#### 5.1.3.0 Limitations of the Normal Equation

The Normal Equation (or 'ordinary least squares' formula) that we used in the above examples relies on linear algebra to resolve a set of equations in a fast and efficient way, allowing us to find an exact solution to the least squares problem. However, some conditions need to be met for the system to be solvable. For instance, this approach only works with square matrices, where you have the same number of unknowns as equations. This is rarely efficient in cases where you have systems with hundreds of dimensions!

One of the major concerns with machine learning and numerical methods in general is optimization using different solvers: i.e., finding the best approximation of a solution for any equation. One such approach that can be used with linear models is called **Gradient Descent**. Let's introduce it.

In [ ]:
from IPython.display import IFrame
display(IFrame('https://youtube.com/embed/l6mTzbPOWHE', width = 640, height = 360))

Gradient descent is a machine learning algorithm that converges by iterating through the dataset. Each of these iterations is known as an *epoch*. The gradient descent algorithm will eventually converge towards the best solution for any linear problem we throw at it. The performance of the algorithm at each epoch is assessed by computing a **cost function**, and the goal of the algorithm is to minimise the value of this cost function. These are terms you will find often in machine learning, **especially in deep-learning**.

For gradient descent functions applied to regression problems, a very common cost function is the MSE that we introduced earlier. At each iteration, both the **value** and the **gradient** cost function are calculated (through calculus and linear algebra) in all of the possible dimensions of your data. The purpose is to determine the direction in which to move the weights and biases such that the cost function will be minimised at the next epoch (or iteration). Once the cost function no longer decreases with additional epochs, the solution is said to have 'converged' and the training stops.

In addition to the weights and biases, a machine learning algorithm also has parameters known as *hyperparameters*. These hyperparameters are options that can be tweaked to control exactly how the algorithm is applied to the dataset. Each type of machine learning algorithm has unique parameters that can be adjusted and optimised. In the case of gradient descent, examples of hyperparameters include the maximum number of iterations through the dataset before you stop training (normally called `max_iter`), the tolerance (i.e., how small of a change in the cost function has to be before you stop training, normally called `tol`), whether or not to apply a penalty to the cost function, and the learning rate (how big a step you take in the direction of the lower gradient at each iteration). It is important to understand that, unlike weights and biases, hyperparameters are **NOT** learned from the data; they are set independently before training the algorithm.

Now that you know more about gradient descent, let's train our first linear model using the gradient descent algorithm. Scikit-learn has specific implementation of the gradient descent algorithm that acts like a linear model: it is known as a 'Stochastic Gradient Descent Regressor' (`SGDRegressor`). `SGDRegressor` is very efficient because it randomly selects 1 sample at each iteration, and upgrades the weights and biases of the algorithm based on this single sample iteration. Consequently, it is extremely efficient and each epoch can be computed extremely quickly. Furthermore, SGD still tends to converge to the right solution, though it requires more iterations than the version of gradient descent that uses all of the samples at each iteration (also known as *Batch Gradient Descent*). Despite this, SGD is still overall faster than Batch Gradient Descent because the time needed to calculate an iteration in SGD is much faster than BGD.

Let's import `SGDRegressor` and use it on our dataset. We will set the following hyperparameters:
* Maximum Number of Iterations (max_iter): 50
* Tolerance (tol): - 0; i.e. we won't stop training based on tolerance
* Penalty: None
* Initial training rate: 0.1

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd_reg = SGDRegressor(max_iter = 50, tol = 0, penalty = None, eta0 = 0.1, random_state = 42)

# Following best practice, let's define a pipeline:

sgd_pipe = Pipeline([('imputer', SimpleImputer()),
                    ('scaler', MinMaxScaler()),
                    ('regressor',sgd_reg)])

sgd_pipe.fit(X_train, y_train)

In [ ]:
# just like before, we can predict from our test:
porosity_predictions_SGD = sgd_pipe.predict(X_test)
porosity_predictions_SGD

### Accessing the trained algorithm in the pipeline

A quick reminder: a pipeline is similar to a Python dictionary. Any item in the pipeline can be accessed by using the name you gave to the object when you created the pipeline. For instance, if we wanted to extract the regressor object from the `sgd_pipe` pipeline, we would simply do that:

In [ ]:
sgd_pipe['regressor']

Similarly, we can extract the imputer or the scaler object if we wanted to:

In [ ]:
sgd_pipe['imputer']

# Exercise 4

Now that you know how to extract specific bits of a trained pipeline, do the following:

1. Create a dataframe that contains 6 columns and 2 rows. The six columns will be the intercepts calculated by the machine learning algorithms: these are the coefficients given to each feature of the model. We will have two rows because we want to compare the values returned by the `LinearRegression` pipeline and the `SGDRegressor` pipeline. Make sure that you use the column names from the original X_train dataset for your coefficients; the coefficients returned by the `.coef_` attribute are in the same order as in the training data.
2. Check whether the intercept and coefficients are similar between the two models: does the SGD approximation do a good job at capturing the coefficients identified by the OLS model?
3. Based on the values of the coefficients, can you tell which of the 5 features we have used to predict porosity is the most important? Remember that because our data is scaled (i.e., all values are between 0 and 1) the coefficient is a direct indication of the relative importance of each feature. If you were to keep only 2 features to retrain a predictive model of porosity, which features would you keep?

In [ ]:
# Your code here

### Conclusions for the Exercise

<details>
<summary>Click here</summary>

The solutions are quite similar between the two models.

Depth and NGR counts are negatively correlated with porosity and are the two strongest indicators for the target value. If we wanted to simplify our model, we could just use these two features and still retain a significant amount of predictive strength.

</details>

### Move on to the next notebook in this class